# PhishGuard-RAG: Train, Test, and Validation Pipeline
TF-IDF Feature Extraction + Logistic Regression (Hyperparameter Tuning on C)

## CELL 1: DATA LOADING AND PREPROCESSING

In [ ]:
# Install required libraries
!pip install pandas numpy scikit-learn matplotlib seaborn joblib -q

# Import libraries
import pandas as pd
import numpy as np
import re
import os
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from google.colab import files

# ------------------------------------------
# 1. Upload both datasets
# ------------------------------------------

uploaded = files.upload()

# Read the uploaded CSV files
df_a = pd.read_csv("phishing_email.csv")
df_b = pd.read_csv("phishing_legit_dataset_KD_10000.csv")

print("Dataset A Shape:", df_a.shape)
print("Dataset B Shape:", df_b.shape)

# ------------------------------------------
# 2. Clean the datasets
# ------------------------------------------

def clean_dataset(df, text_column, label_column):

    # Keep only the required columns
    df = df[[text_column, label_column]].copy()

    # Rename columns for consistency
    df.columns = ["text", "label"]

    # Remove missing values
    df = df.dropna(subset=["text", "label"])

    # Convert email text into string
    df["text"] = df["text"].astype(str)

    # Normalize extra spaces and new lines
    df["text"] = df["text"].apply(
        lambda x: re.sub(r"\s+", " ", x).strip()
    )

    # Remove empty messages
    df = df[df["text"] != ""]

    # Check for duplicate text with conflicting labels
    conflicting = (
        df.groupby("text")["label"].nunique()
    )

    conflicting_texts = conflicting[conflicting > 1].index

    print("Conflicting duplicate messages:", len(conflicting_texts))

    # Remove conflicting messages to avoid ambiguous labels
    df = df[~df["text"].isin(conflicting_texts)]

    # Remove exact duplicate messages
    df = df.drop_duplicates(subset=["text"])

    # Ensure labels are binary integers
    df["label"] = pd.to_numeric(df["label"], errors="raise")

    if not set(df["label"].unique()).issubset({0, 1}):
        raise ValueError("Labels must contain only 0 and 1.")

    df["label"] = df["label"].astype(int)

    return df.reset_index(drop=True)


# Clean Dataset A
data_a = clean_dataset(
    df_a,
    text_column="text_combined",
    label_column="label"
)

# Clean Dataset B
data_b = clean_dataset(
    df_b,
    text_column="text",
    label_column="label"
)

print("\nCleaned Dataset A:", data_a.shape)
print("Cleaned Dataset B:", data_b.shape)

# ------------------------------------------
# 3. Display class distribution
# ------------------------------------------

print("\nDataset A Class Distribution:")
print(data_a["label"].value_counts())

print("\nDataset B Class Distribution:")
print(data_b["label"].value_counts())

# ------------------------------------------
# 4. Split into train, validation and test
# ------------------------------------------

def split_dataset(df):

    X = df["text"]
    y = df["label"]

    # First separate 20% for final testing
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    # Separate 10% of total data for validation
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=0.125,
        random_state=42,
        stratify=y_train_val
    )

    return X_train, X_val, X_test, y_train, y_val, y_test


# Split Dataset A
A_train, A_val, A_test, Ay_train, Ay_val, Ay_test = split_dataset(data_a)

# Split Dataset B
B_train, B_val, B_test, By_train, By_val, By_test = split_dataset(data_b)

# ------------------------------------------
# 5. Display split sizes
# ------------------------------------------

print("\nDATASET A SPLIT")
print("Training:", len(A_train))
print("Validation:", len(A_val))
print("Testing:", len(A_test))

print("\nDATASET B SPLIT")
print("Training:", len(B_train))
print("Validation:", len(B_val))
print("Testing:", len(B_test))

print("\nPreprocessing and splitting completed!")


## CELL 2: TRAINING, VALIDATION AND TESTING

In [ ]:
import time

# ------------------------------------------
# 1. Function to train and evaluate a model
# ------------------------------------------

def train_and_evaluate(
    X_train, X_val, X_test,
    y_train, y_val, y_test,
    dataset_name
):

    print("\n" + "=" * 55)
    print("RESULTS FOR", dataset_name)
    print("=" * 55)

    # --------------------------------------
    # 2. Convert text into TF-IDF features
    # --------------------------------------

    vectorizer = TfidfVectorizer(
        max_features=30000,
        ngram_range=(1, 2),
        stop_words=None,
        min_df=2,
        sublinear_tf=True
    )

    # Fit only on training data
    X_train_tfidf = vectorizer.fit_transform(X_train)

    # Transform validation data
    X_val_tfidf = vectorizer.transform(X_val)

    # --------------------------------------
    # 3. Hyperparameter tuning
    # --------------------------------------

    C_values = [0.1, 0.5, 1, 2, 5, 10]

    best_model = None
    best_C = None
    best_f1 = -1

    print("\nVALIDATION RESULTS")

    for c in C_values:

        model = LogisticRegression(
            C=c,
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )

        # Train using training data
        model.fit(X_train_tfidf, y_train)

        # Predict validation data
        val_predictions = model.predict(X_val_tfidf)

        # Calculate validation F1-score
        val_f1 = f1_score(
            y_val,
            val_predictions,
            zero_division=0
        )

        print(f"C = {c} | Validation F1 = {val_f1:.4f}")

        # Save the best model
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_C = c
            best_model = model

    print("\nBest C:", best_C)
    print("Best Validation F1:", round(best_f1, 4))

    # --------------------------------------
    # 4. Refit using training + validation
    # --------------------------------------

    X_train_final = pd.concat([X_train, X_val])
    y_train_final = pd.concat([y_train, y_val])

    # Create a fresh vectorizer
    final_vectorizer = TfidfVectorizer(
        max_features=30000,
        ngram_range=(1, 2),
        stop_words=None,
        min_df=2,
        sublinear_tf=True
    )

    X_train_final_tfidf = final_vectorizer.fit_transform(
        X_train_final
    )

    # Train final model with the selected C
    final_model = LogisticRegression(
        C=best_C,
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )

    final_model.fit(X_train_final_tfidf, y_train_final)

    # --------------------------------------
    # 5. Final testing
    # --------------------------------------

    X_test_tfidf = final_vectorizer.transform(X_test)

    start_time = time.time()

    test_predictions = final_model.predict(X_test_tfidf)

    prediction_time = time.time() - start_time

    # --------------------------------------
    # 6. Calculate evaluation metrics
    # --------------------------------------

    accuracy = accuracy_score(y_test, test_predictions)

    precision = precision_score(
        y_test,
        test_predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        test_predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        test_predictions,
        zero_division=0
    )

    print("\nFINAL TEST RESULTS")
    print("------------------------------")
    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))
    print("Prediction Time:", round(prediction_time, 4), "seconds")

    # Detailed classification report
    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            test_predictions,
            target_names=["Legitimate", "Phishing"],
            zero_division=0
        )
    )

    # --------------------------------------
    # 7. Confusion Matrix
    # --------------------------------------

    cm = confusion_matrix(y_test, test_predictions)

    plt.figure(figsize=(6, 4))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Legitimate", "Phishing"],
        yticklabels=["Legitimate", "Phishing"]
    )

    plt.xlabel("Predicted Label")
    plt.ylabel("Actual Label")
    plt.title(dataset_name + " - Confusion Matrix")
    plt.show()

    # --------------------------------------
    # 8. Save trained model and vectorizer
    # --------------------------------------

    model_filename = dataset_name.lower().replace(" ", "_")

    joblib.dump(
        final_model,
        model_filename + "_model.pkl"
    )

    joblib.dump(
        final_vectorizer,
        model_filename + "_vectorizer.pkl"
    )

    # Return results
    return {
        "Dataset": dataset_name,
        "Best C": best_C,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Prediction Time (seconds)": prediction_time
    }


# ------------------------------------------
# 9. Train and test Dataset A
# ------------------------------------------

results_a = train_and_evaluate(
    A_train, A_val, A_test,
    Ay_train, Ay_val, Ay_test,
    "Dataset A"
)


# ------------------------------------------
# 10. Train and test Dataset B
# ------------------------------------------

results_b = train_and_evaluate(
    B_train, B_val, B_test,
    By_train, By_val, By_test,
    "Dataset B"
)


# ------------------------------------------
# 11. Compare both datasets
# ------------------------------------------

results_df = pd.DataFrame([
    results_a,
    results_b
])

print("\nFINAL COMPARISON")
display(results_df)

# Save comparison results
results_df.to_csv(
    "phishguard_results.csv",
    index=False
)

print("\nAll training and testing completed!")

# ------------------------------------------
# 12. Download saved files
# ------------------------------------------

files.download("phishguard_results.csv")
files.download("dataset_a_model.pkl")
files.download("dataset_a_vectorizer.pkl")
files.download("dataset_b_model.pkl")
files.download("dataset_b_vectorizer.pkl")
